In [1]:
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/   # upload your kaggle.json first
!chmod 600 ~/.kaggle/kaggle.json

!kaggle kernels output ryanmfleishman/dl-llama-version -p /content/model

chat_template.jinja: Skipping, found more recently modified local copy (use --force to force download)
2f2f7ed8cf1b5a91f2bbb383cb767e679c0a6aed: Skipping, found more recently modified local copy (use --force to force download)
31349551d90c7606f325fe0f11bbb8bd5fa0d7c7: Skipping, found more recently modified local copy (use --force to force download)
4783fe10ac3adce15ac8f358ef5462739852c569: Skipping, found more recently modified local copy (use --force to force download)
630e242dd9b9ba3ac98f68aecd23f65ec95efa7e: Skipping, found more recently modified local copy (use --force to force download)
6525ab272147015106fe845559652461ccc4cc84: Skipping, found more recently modified local copy (use --force to force download)
c8dd4c904ea9bfc3cbaaa50075c13a9a1dab52c9: Skipping, found more recently modified local copy (use --force to force download)
fefe97458e88e3dab707364053da1584688dc4ca: Skipping, found more recently modified local copy (use --force to force download)
main: Skipping, found more re

In [2]:
!pip install -q unsloth transformers accelerate peft bitsandbytes

In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/model/qwen25_coder_svg_lora_merged",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/model/qwen25_coder_svg_lora_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/model/qwen25_coder_svg_lora_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RM

In [4]:
FastLanguageModel.for_inference(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RM

In [5]:
import re
import torch
import xml.etree.ElementTree as ET

# Regex to extract SVG
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""

def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False

def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

SYSTEM_PROMPT = "You are an expert SVG generator. Output only valid SVG code."

def generate_svg(prompt, max_new_tokens=512):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)

    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)

    return svg

In [6]:
test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

<svg width="100" height="100" viewBox="0 0 100 100" xmlns="http://www.w3.org/2000/svg">
  <circle cx="50" cy="50" r="40" fill="#007bff"/>
  <path d="M50 50 Q50 65 65 50 Q50 35 50 50" stroke="#007bff" stroke-width="4"/>
  <path d="M50 75 Q50 90 65 75 Q50 60 50 75" stroke="#007bff" stroke-width="4"/>
</svg>
Valid SVG: True


In [7]:
import re
import torch
import xml.etree.ElementTree as ET

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)
SVG_START_REGEX = re.compile(r"<svg\b[\s\S]*", flags=re.IGNORECASE)

def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""

def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False

def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

SYSTEM_PROMPT = (
    "You are an expert SVG generator. "
    "Return only valid SVG markup. "
    "Do not include markdown fences, explanations, or any text before or after the SVG."
)

def generate_raw_output(prompt, max_new_tokens=768):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

def repair_svg_from_raw(text):
    svg = extract_svg(text)
    if is_valid_svg(svg):
        return svg

    m = SVG_START_REGEX.search(text)
    if not m:
        return ""

    partial = m.group(0).strip()

    # remove unfinished trailing fragment like "<path d=..."
    last_lt = partial.rfind("<")
    last_gt = partial.rfind(">")
    if last_lt > last_gt:
        partial = partial[:last_lt]

    # add closing tag if missing
    if "</svg>" not in partial.lower():
        partial += "</svg>"

    try:
        ET.fromstring(partial)
        return partial
    except ET.ParseError:
        return ""

def generate_svg_with_repair(prompt, tries=3, max_new_tokens=768):
    attempts = []

    for _ in range(tries):
        raw = generate_raw_output(prompt, max_new_tokens=max_new_tokens)
        repaired = repair_svg_from_raw(raw)
        valid = is_valid_svg(repaired)

        attempts.append({
            "raw": raw,
            "repaired": repaired,
            "valid": valid,
        })

        if valid:
            return repaired, attempts

    return fallback_svg(prompt), attempts

In [8]:
def has_meaningful_content(svg_text):
    if not svg_text or not is_valid_svg(svg_text):
        return False
    try:
        root = ET.fromstring(svg_text)
        meaningful_tags = {
            "path", "rect", "circle", "ellipse", "line",
            "polyline", "polygon", "text", "g"
        }
        count = 0
        for elem in root.iter():
            tag = elem.tag.split("}")[-1].lower()
            if tag in meaningful_tags:
                count += 1
        return count >= 2
    except ET.ParseError:
        return False

def generate_svg_with_repair(prompt, tries=3, max_new_tokens=768):
    attempts = []

    for _ in range(tries):
        raw = generate_raw_output(prompt, max_new_tokens=max_new_tokens)
        repaired = repair_svg_from_raw(raw)
        valid = is_valid_svg(repaired)
        meaningful = has_meaningful_content(repaired)

        attempts.append({
            "raw": raw,
            "repaired": repaired,
            "valid": valid,
            "meaningful": meaningful,
        })

        if valid and meaningful:
            return repaired, attempts

    return fallback_svg(prompt), attempts

In [10]:
import pandas as pd
test_df = pd.read_csv("/content/test.csv")
print(test_df.columns)
print(test_df.head())

Index(['id', 'prompt'], dtype='object')
                                     id  \
0  fa1d8fa7-080f-4269-a9cf-a17562c9a0ca   
1      6eede943219547c22ac56085027d33cc   
2      ea045c7a247166f061ce504d9b7ccaab   
3      8fe82f3af89e487b31236ca829c3f071   
4      600464e4d92c75338462271a09b3f176   

                                              prompt  
0  firewood stack cut logs wood with leaf illustr...  
1  The image shows five horizontal lines of varyi...  
2  A stylized icon depicting a curved arrow withi...  
3  The image contains black geometric shapes agai...  
4  The image shows a single dark gray triangle po...  


In [11]:
mini_predictions = []
mini_fallbacks = 0

for i, prompt in enumerate(test_df["prompt"].head(10)):
    svg, attempts = generate_svg_with_repair(prompt, tries=3, max_new_tokens=768)

    if svg == fallback_svg(prompt):
        mini_fallbacks += 1

    mini_predictions.append(svg)
    print(f"{i}: valid={is_valid_svg(svg)} fallback={svg == fallback_svg(prompt)}")
    print(svg[:200])
    print("-" * 80)

print("Mini fallback count:", mini_fallbacks)

Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=7

0: valid=True fallback=False
<svg viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg">
  <path d="m168 40c-7.53 0-14 6.47-14 14v28c0 7.53 6.47 14 14 14h24v-28c0-7.53-6.47-14-14-14z" fill="#2d2d2d"/>
  <path d="m168 72c-7.53 
--------------------------------------------------------------------------------


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


1: valid=True fallback=False
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 300 100" width="300px" height="100px">
  <g fill="#FFFFFF">
    <!-- Line 1 -->
    <rect x="50" y="20" width="20" height="40" transform="rotate(45
--------------------------------------------------------------------------------


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

7: valid=True fallback=False
<svg viewBox="0 0 256 256" width="128" height="128" xmlns="http://www.w3.org/2000/svg">
  <g fill="none" stroke="#000000" stroke-width="1.5">
    <!-- Eye -->
    <circle cx="112.5" cy="112.5" r="10" 
--------------------------------------------------------------------------------


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


8: valid=True fallback=False
<svg viewBox="0 0 100 100" xmlns="http://www.w3.org/2000/svg">
  <rect width="100" height="100" fill="#fff"/>
  <path d="M50 90 L40 80 L60 70" stroke="#000000" stroke-width="2" fill="none"/>
  <path d
--------------------------------------------------------------------------------


Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


9: valid=True fallback=False
<svg viewBox="0 0 32 32" xmlns="http://www.w3.org/2000/svg">
  <rect width="32" height="32" fill="#ffffff"/>
  <path d="m16 19c-4.5 0-8-3.5-8-8s3.5-8 8-8 8 3.5 8 8-3.5 8-8 8z" fill="#000000"/>
  <path
--------------------------------------------------------------------------------
Mini fallback count: 0


In [ ]:
predictions = []
num_fallbacks = 0

for i, prompt in enumerate(test_df["prompt"]):
    print(f"Starting {i}")

    svg, attempts = generate_svg_with_repair(prompt, tries=3, max_new_tokens=512)

    print(f"Finished {i} | fallback={svg == fallback_svg(prompt)}")

    if svg == fallback_svg(prompt):
        num_fallbacks += 1

    predictions.append(svg)

    if (i + 1) % 25 == 0:
        print(f"Done {i+1}/{len(test_df)} | fallbacks so far: {num_fallbacks}")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting 0


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished 0 | fallback=False
Starting 1


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished 29 | fallback=False
Starting 30


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished 30 | fallback=False
Starting 31


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print("Fallback count:", num_fallbacks)
print("All valid:", all(is_valid_svg(s) for s in predictions))

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "svg": predictions,
})

submission.to_csv("/content/submission.csv", index=False)
print("Saved /content/submission.csv")